# GR00T LIBERO Colab L4 Notebook

This notebook runs `nvidia/GR00T-N1.7-LIBERO` on LIBERO with the same Colab controls as the Pi0.5 notebook, so teammates can compare the two models side by side.

Open this file from the `groot-run` branch:

```text
https://colab.research.google.com/github/MarquiseRosier/pi05-run/blob/groot-run/groot-run/notebooks/groot_libero_colab_l4.ipynb
```

Pi0.5 counterpart on `main`: `notebooks/pi05_libero_colab_l4.ipynb`.

Target workflow:

1. Open this notebook in Colab.
2. Runtime -> Change runtime type -> GPU -> **L4** or **A100**, preferably high-RAM. Do not use T4.
3. Mount the shared Drive folder. GR00T caches live next to the Pi0.5 archives, not inside them.
4. Runtime -> Run all.
5. Inspect rollout videos and the activation report inline.

This is not a copy of the Pi0.5 eval process. GR00T uses two processes: a GPU policy server and a LIBERO sim client on ZMQ port 5555.

Activation families are remapped onto GR00T N1.7 (`vision` = backbone vision, `prefix` = backbone language, `expert` = DiT action head, `projection` = action/state encoders). The HTML report UX matches Pi0.5.


In [ ]:
# @title Controls

# Drive folder shared only with approved collaborators.
DRIVE_ROOT = "/content/drive/MyDrive/groot-run-shared-programmer908"  # @param {type:"string"}
DRIVE_FOLDER_ID = "13z_Qh91Ww0jdXoju2kQsG12AaEPKZsnH"  # @param {type:"string"}
AUTO_CREATE_DRIVE_SHORTCUT = True  # @param {type:"boolean"}
APPROVED_ACCOUNTS = "programmer908@gmail.com"  # @param {type:"string"}
SHARED_HF_TOKEN_FILE = "secrets/HF_TOKEN.txt"  # @param {type:"string"}

# Repository.
REPO_URL = "https://github.com/MarquiseRosier/pi05-run.git"  # @param {type:"string"}
REPO_BRANCH = "groot-run"  # @param {type:"string"}

# Runtime preference. Match the Pi0.5 notebook so both models use the same machine class.
REQUIRED_GPU = "L4"  # @param ["L4", "A100", "Any"]
MIN_GPU_MEMORY_GB = 20  # @param {type:"integer"}
MIN_SYSTEM_RAM_GB = 24  # @param {type:"integer"}

# Evaluation controls. Same names as the Pi0.5 notebook.
SUITE = "libero_spatial"  # @param ["libero_spatial", "libero_object", "libero_goal", "libero_10", "libero_spatial,libero_object,libero_goal,libero_10"]
TASK_IDS = "[0]"  # @param {type:"string"}
ANALYSIS_TASK_ID = 0  # @param {type:"integer"}
EPISODES = 1  # @param {type:"integer"}
EVAL_PROGRESS_SECONDS = 30  # @param {type:"integer"}
N_ACTION_STEPS = 8  # @param {type:"integer"}
N_ENVS = 1  # @param {type:"integer"}
MAX_EPISODE_STEPS = 720  # @param {type:"integer"}

# Checkpoint. Download into Drive archives or local checkpoints/.
MODEL_PATH = "checkpoints/GR00T-N1.7-LIBERO/libero_10"  # @param {type:"string"}
EMBODIMENT_TAG = "LIBERO_PANDA"  # @param {type:"string"}

# Activation capture controls.
CAPTURE_ACTIVATIONS = True  # @param {type:"boolean"}
CAPTURE_PARAM_STATS = False  # @param {type:"boolean"}
CAPTURE_MAX_CHUNKS = 40  # @param {type:"integer"}
CAPTURE_LAYER_STRIDE = 1  # @param {type:"integer"}
CAPTURE_MAX_BINS = 64  # @param {type:"integer"}

# Colab report controls.
REPORT_MAX_ROWS = 80  # @param {type:"integer"}
GENERATE_DIAGNOSTIC_VIDEO = False  # @param {type:"boolean"}
DISPLAY_INDIVIDUAL_LAYER_GRAPHS = False  # @param {type:"boolean"}
LAYER_GRAPH_LIMIT = 6  # @param {type:"integer"}

# Cache behavior. Archive mode avoids slow Google Drive small-file copies.
CACHE_TRANSFER_MODE = "archive"  # @param ["archive", "folders"]
ALLOW_AUTH_REFRESH = True  # @param {type:"boolean"}
FORCE_AUTH_REFRESH = False  # @param {type:"boolean"}
# False until archives/groot_ckpt.tar exists. Pi0.5's archives/hf_home.tar is not GR00T.
HF_OFFLINE = False  # @param {type:"boolean"}

# Language intervention probe. PROBE_LANGUAGE must differ from the task prompt.
PROBE_SUITE = "libero_spatial"  # @param ["libero_spatial", "libero_object", "libero_goal", "libero_10"]
PROBE_TASK_ID = 0  # @param {type:"integer"}
PROBE_LANGUAGE = "pick up the cookie box and place it on the plate"  # @param {type:"string"}
PROBE_SEED = 1000  # @param {type:"integer"}
PROBE_N_REPLANS = 4  # @param {type:"integer"}

print("Configured approved accounts:", APPROVED_ACCOUNTS)
print("Repo branch:", REPO_BRANCH)
print("Suite / tasks / episodes:", SUITE, TASK_IDS, EPISODES)


In [ ]:
# @title Mount Drive And Validate Runtime

from pathlib import Path
import os
import subprocess
import time

from google.colab import drive


def parse_gib_from_meminfo() -> int:
    with open("/proc/meminfo", "r", encoding="utf-8") as handle:
        for line in handle:
            if line.startswith("MemTotal:"):
                kb = int(line.split()[1])
                return (kb + 1024 * 1024 - 1) // (1024 * 1024)
    return 0


def validate_colab_runtime():
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader,nounits"],
        text=True,
        capture_output=True,
    )
    print(result.stdout or result.stderr)
    if result.returncode != 0:
        raise RuntimeError("No NVIDIA GPU visible. In Colab: Runtime -> Change runtime type -> GPU.")

    gpu_line = result.stdout.strip().splitlines()[0]
    parts = [part.strip() for part in gpu_line.split(",")]
    gpu_name = parts[0]
    gpu_mem_gb = int((int(parts[2]) + 1023) // 1024) if len(parts) >= 3 and parts[2].isdigit() else 0
    system_ram_gb = parse_gib_from_meminfo()
    print(f"Runtime memory: GPU={gpu_name} ~{gpu_mem_gb} GiB VRAM | system RAM ~{system_ram_gb} GiB")

    if REQUIRED_GPU != "Any" and REQUIRED_GPU.lower() not in gpu_name.lower():
        raise RuntimeError(
            f"Requested {REQUIRED_GPU}, but Colab allocated {gpu_name}. "
            "Use Runtime -> Change runtime type -> GPU -> L4/A100, then rerun from this cell."
        )
    if int(MIN_GPU_MEMORY_GB) > 0 and gpu_mem_gb < int(MIN_GPU_MEMORY_GB):
        raise RuntimeError(
            f"GPU VRAM is too small for GR00T LIBERO: {gpu_name} has ~{gpu_mem_gb} GiB, "
            f"need >= {MIN_GPU_MEMORY_GB} GiB. T4 is not enough. Switch to L4 or A100."
        )
    if int(MIN_SYSTEM_RAM_GB) > 0 and system_ram_gb < int(MIN_SYSTEM_RAM_GB):
        raise RuntimeError(
            f"System RAM is too small for GR00T LIBERO: runtime has ~{system_ram_gb} GiB, "
            f"need >= {MIN_SYSTEM_RAM_GB} GiB. In Colab, choose a high-RAM L4/A100 runtime."
        )


validate_colab_runtime()
drive.mount("/content/drive")


def ensure_drive_root_visible(root_value: str) -> Path:
    root = Path(root_value)
    if root.exists():
        return root

    folder_id = str(globals().get("DRIVE_FOLDER_ID", "")).strip()
    auto_shortcut = bool(globals().get("AUTO_CREATE_DRIVE_SHORTCUT", True))
    mydrive_prefixes = ("/content/drive/MyDrive/", "/content/drive/My Drive/")
    if folder_id and auto_shortcut and str(root).startswith(mydrive_prefixes):
        print("Drive root is not visible yet; creating a My Drive shortcut to the shared folder.", flush=True)
        try:
            from google.colab import auth
            auth.authenticate_user()
            try:
                from googleapiclient.discovery import build
            except Exception:
                subprocess.run(["python3", "-m", "pip", "install", "-q", "-U", "google-api-python-client"], check=True)
                from googleapiclient.discovery import build
            service = build("drive", "v3")
            metadata = {
                "name": root.name,
                "mimeType": "application/vnd.google-apps.shortcut",
                "shortcutDetails": {"targetId": folder_id},
                "parents": ["root"],
            }
            created = service.files().create(body=metadata, fields="id,name").execute()
            print(f"Created Drive shortcut: {created.get('name')} ({created.get('id')})", flush=True)
            for _ in range(12):
                if root.exists():
                    return root
                time.sleep(5)
        except Exception as exc:
            raise RuntimeError(
                "Could not auto-create the Drive shortcut. Confirm the folder was shared with this Google account "
                "and DRIVE_FOLDER_ID is the shared folder ID."
            ) from exc
        if root.exists():
            return root
        raise RuntimeError(f"Drive shortcut was created, but {root} is not visible yet. Rerun this cell.")
    if not folder_id:
        print("DRIVE_FOLDER_ID is blank; creating DRIVE_ROOT if needed.", flush=True)
    return root


DRIVE_ROOT = ensure_drive_root_visible(DRIVE_ROOT)
DRIVE_ARCHIVES = DRIVE_ROOT / "archives"
DRIVE_HF_HOME = DRIVE_ROOT / "groot_hf_home"
DRIVE_OUTPUTS = DRIVE_ROOT / "outputs"
DRIVE_NOTEBOOK_META = DRIVE_ROOT / "notebook_meta"
DRIVE_SECRETS = DRIVE_ROOT / "secrets"
DRIVE_HF_TOKEN_FILE = DRIVE_ROOT / SHARED_HF_TOKEN_FILE
for path in [DRIVE_ROOT, DRIVE_ARCHIVES, DRIVE_HF_HOME, DRIVE_OUTPUTS, DRIVE_NOTEBOOK_META, DRIVE_SECRETS]:
    path.mkdir(parents=True, exist_ok=True)

HF_HOME_ARCHIVE = DRIVE_ARCHIVES / "groot_hf_home.tar"
CKPT_ARCHIVE = DRIVE_ARCHIVES / "groot_ckpt.tar"
print("Drive root:", DRIVE_ROOT)
print("GR00T HF archive:", HF_HOME_ARCHIVE)
print("GR00T ckpt archive:", CKPT_ARCHIVE)
print("Expected sharing: restricted to", APPROVED_ACCOUNTS)


In [ ]:
# @title Install Native Runtime Tools

from pathlib import Path
import os
import subprocess

GROOT_VENV = Path("/content/groot-venv")
GROOT_PYTHON = GROOT_VENV / "bin/python"
LIBERO_VENV = Path("/content/libero_uv/.venv")
LIBERO_PYTHON = LIBERO_VENV / "bin/python"
UV_BIN_DIR = Path("/content/uv-bin")
UV = str(UV_BIN_DIR / "uv")


def run(cmd, *, env=None, cwd=None):
    cmd = list(map(str, cmd))
    print("$", " ".join(cmd), flush=True)
    result = subprocess.run(cmd, env=env, cwd=cwd, text=True, capture_output=False)
    if result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, cmd)
    return result


def install_uv() -> str:
    UV_BIN_DIR.mkdir(parents=True, exist_ok=True)
    installer = Path("/tmp/install-uv.sh")
    run(["curl", "-LsSf", "https://astral.sh/uv/install.sh", "-o", installer])
    env = os.environ.copy()
    env["UV_INSTALL_DIR"] = str(UV_BIN_DIR)
    run(["sh", installer], env=env)
    os.environ["PATH"] = f"{UV_BIN_DIR}:" + os.environ["PATH"]
    run([UV, "--version"])
    return UV


apt_packages = [
    "build-essential", "ca-certificates", "cmake", "curl", "ffmpeg", "git", "pv", "rsync",
    "libegl1", "libgl1", "libglib2.0-0", "libglvnd0", "libglx0", "libopengl0",
    "libosmesa6-dev", "libsm6", "libxext6", "libxrender1", "pkg-config",
]
apt_env = os.environ.copy()
apt_env["DEBIAN_FRONTEND"] = "noninteractive"
run(["apt-get", "update", "-qq"], env=apt_env)
run(["apt-get", "install", "-y", "-qq", *apt_packages], env=apt_env)
UV = install_uv()
run([UV, "python", "install", "3.12"])
print("uv ready:", UV)


In [ ]:
# @title Clone Or Update Repo

from pathlib import Path
import subprocess

LOCAL_REPO = Path("/content/pi05-run")
if LOCAL_REPO.exists():
    subprocess.run(["git", "-C", str(LOCAL_REPO), "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(LOCAL_REPO), "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(LOCAL_REPO), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(LOCAL_REPO)], check=True)

ISAAC_ROOT = LOCAL_REPO / "MI_VLA" / "Isaac-GR00T"
GROOT_RUN = LOCAL_REPO / "groot-run"
if not ISAAC_ROOT.exists():
    raise RuntimeError(f"MI_VLA/Isaac-GR00T missing in {LOCAL_REPO}. Push that tree on the groot-run branch.")
print("Repo:", LOCAL_REPO)
subprocess.run(["git", "-C", str(LOCAL_REPO), "rev-parse", "--short", "HEAD"], check=True)
print("Isaac-GR00T:", ISAAC_ROOT)


In [ ]:
# @title Install GR00T And LIBERO Venvs

from pathlib import Path
import os
import subprocess

# Two venvs on purpose. Do not merge GR00T and LIBERO dependencies.
if not GROOT_PYTHON.exists():
    run([UV, "venv", "--clear", str(GROOT_VENV), "--python", "3.12"])
    run([
        UV, "pip", "install", "--python", str(GROOT_PYTHON), "--torch-backend", "cu128",
        "-e", str(ISAAC_ROOT), "hf-transfer", "opencv-python", "numpy",
    ])

setup_libero = ISAAC_ROOT / "gr00t/eval/sim/LIBERO/setup_libero.sh"
native_libero = ISAAC_ROOT / "gr00t/eval/sim/LIBERO/libero_uv/.venv/bin/python"
os.environ["PATH"] = f"{UV_BIN_DIR}:{os.environ.get('PATH', '')}"
os.environ["UV_INSTALL_DIR"] = str(UV_BIN_DIR)
def _libero_venv_ready(python_path) -> bool:
    if not python_path.exists():
        return False
    check = subprocess.run(
        [str(python_path), "-c", "import numpy, gymnasium, libero"],
        capture_output=True,
        text=True,
    )
    return check.returncode == 0


if _libero_venv_ready(native_libero):
    print("Using repo LIBERO venv:", native_libero)
    LIBERO_PYTHON = native_libero
elif _libero_venv_ready(LIBERO_PYTHON):
    print("Using /content LIBERO venv:", LIBERO_PYTHON)
else:
    run(
        ["bash", str(GROOT_RUN / "scripts/setup_libero_colab.sh"), str(ISAAC_ROOT)],
        cwd=str(ISAAC_ROOT),
        env=os.environ.copy(),
    )
    if _libero_venv_ready(native_libero):
        LIBERO_PYTHON = native_libero
    elif not _libero_venv_ready(LIBERO_PYTHON):
        raise RuntimeError("LIBERO venv was not created by setup_libero.sh")

os.environ["PATH"] = f"{GROOT_VENV / 'bin'}:{UV_BIN_DIR}:" + os.environ["PATH"]
print("GROOT_PYTHON:", GROOT_PYTHON)
print("LIBERO_PYTHON:", LIBERO_PYTHON)
run([str(GROOT_PYTHON), "-c", "import torch; print('torch', torch.__version__); print('cuda', torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no gpu')"])


In [ ]:
# @title Prepare Drive Caches And Offline Mode

from pathlib import Path
import json
import os
import shutil
import subprocess
import time

LOCAL_HF_HOME = Path("/content/groot_hf_home")
LOCAL_OUTPUT_ROOT = LOCAL_REPO / "outputs/eval/groot_libero"
LOCAL_CKPT_ROOT = ISAAC_ROOT / "checkpoints"
LOCAL_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_CKPT_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_HF_HOME.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(LOCAL_HF_HOME)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
if HF_OFFLINE:
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"
else:
    os.environ.pop("HF_HUB_OFFLINE", None)
    os.environ.pop("TRANSFORMERS_OFFLINE", None)


def resolve_hf_token() -> str:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            return token
    except Exception:
        pass
    if DRIVE_HF_TOKEN_FILE.exists():
        return DRIVE_HF_TOKEN_FILE.read_text().strip()
    return os.environ.get("HF_TOKEN", "")


def extract_archive(archive: Path, dest: Path) -> None:
    if not archive.exists():
        print("missing archive:", archive)
        return
    dest.mkdir(parents=True, exist_ok=True)
    print(f"Extracting {archive} -> {dest}", flush=True)
    subprocess.run(f"pv {archive} | tar -xf - -C {dest}", shell=True, check=True)


pi05_archive = DRIVE_ARCHIVES / "hf_home.tar"
if pi05_archive.exists() and not CKPT_ARCHIVE.exists():
    print("Note:", pi05_archive.name, "is the Pi0.5 cache. GR00T needs archives/groot_ckpt.tar.")

if CACHE_TRANSFER_MODE == "archive":
    if HF_HOME_ARCHIVE.exists() and (FORCE_AUTH_REFRESH or not any(LOCAL_HF_HOME.iterdir())):
        extract_archive(HF_HOME_ARCHIVE, LOCAL_HF_HOME)
    if CKPT_ARCHIVE.exists() and (FORCE_AUTH_REFRESH or not (LOCAL_CKPT_ROOT / "GR00T-N1.7-LIBERO").exists()):
        extract_archive(CKPT_ARCHIVE, LOCAL_CKPT_ROOT)
elif CACHE_TRANSFER_MODE == "folders":
    if DRIVE_HF_HOME.exists():
        subprocess.run(["rsync", "-a", "--info=progress2", f"{DRIVE_HF_HOME}/", f"{LOCAL_HF_HOME}/"], check=True)

model_dir = Path(MODEL_PATH)
if not model_dir.is_absolute():
    model_dir = ISAAC_ROOT / MODEL_PATH
print("Model path:", model_dir)

token = resolve_hf_token()
if token:
    os.environ["HF_TOKEN"] = token

need_refresh = FORCE_AUTH_REFRESH or not model_dir.exists()
if need_refresh and HF_OFFLINE:
    print("HF_OFFLINE=True but GR00T weights are missing; downloading from Hugging Face.")
    os.environ.pop("HF_HUB_OFFLINE", None)
    os.environ.pop("TRANSFORMERS_OFFLINE", None)

if need_refresh or FORCE_AUTH_REFRESH:
    if not token:
        raise RuntimeError(
            "Need HF_TOKEN in Colab Secrets or DRIVE_ROOT/secrets/HF_TOKEN.txt. "
            "Accept nvidia/GR00T-N1.7-LIBERO and nvidia/Cosmos-Reason2-2B, then rerun this cell. "
            "Do not put checkpoints in git."
        )
    ensure = GROOT_RUN / "scripts/ensure_groot_libero_ckpt.py"
    cmd = [str(GROOT_PYTHON), str(ensure), "--dest-root", str(ISAAC_ROOT / "checkpoints/GR00T-N1.7-LIBERO")]
    if FORCE_AUTH_REFRESH:
        cmd.append("--force")
    run(cmd)

print("Local HF home:", LOCAL_HF_HOME)
print("Local checkpoints:", LOCAL_CKPT_ROOT)
print("Model exists:", model_dir.exists())
if not model_dir.exists():
    raise RuntimeError(f"GR00T checkpoint still missing: {model_dir}")
config = json.loads((model_dir / "config.json").read_text(encoding="utf-8"))
if config.get("model_type") != "Gr00tN1d7":
    raise RuntimeError(f"Invalid GR00T config at {model_dir / 'config.json'}: {config.get('model_type')}")
print("model_type:", config["model_type"])
print("First server start will also pull Cosmos-Reason2-2B into HF_HOME.")


In [ ]:
# @title Run LIBERO Eval With Progress

import ast
import json
import os
import re
import subprocess
import threading
import time
from pathlib import Path

run_env = os.environ.copy()
run_id = time.strftime("%Y%m%d-%H%M%S", time.gmtime())
run_dir = LOCAL_OUTPUT_ROOT / run_id
suffix = 1
while run_dir.exists():
    run_id = f"{run_id}-{suffix}"
    run_dir = LOCAL_OUTPUT_ROOT / run_id
    suffix += 1
run_dir.mkdir(parents=True, exist_ok=True)
LATEST_RUN = run_dir

model_dir = Path(MODEL_PATH)
if not model_dir.is_absolute():
    model_dir = ISAAC_ROOT / MODEL_PATH

run_env.update({
    "GROOT_PYTHON": str(GROOT_PYTHON),
    "LIBERO_PYTHON": str(LIBERO_PYTHON),
    "HOST_PYTHON": str(GROOT_PYTHON),
    "ISAAC_ROOT": str(ISAAC_ROOT),
    "MODEL_PATH": str(model_dir),
    "EMBODIMENT_TAG": EMBODIMENT_TAG,
    "TASK_IDS": TASK_IDS.strip(),
    "RUN_ID": run_id,
    "OUTPUT_ROOT": str(LOCAL_OUTPUT_ROOT),
    "N_ACTION_STEPS": str(N_ACTION_STEPS),
    "N_ENVS": str(N_ENVS),
    "MAX_EPISODE_STEPS": str(MAX_EPISODE_STEPS),
    "CAPTURE_ACTIVATIONS": "1" if CAPTURE_ACTIVATIONS else "0",
    "CAPTURE_PARAM_STATS": "1" if CAPTURE_PARAM_STATS else "0",
    "CAPTURE_MAX_CHUNKS": str(CAPTURE_MAX_CHUNKS),
    "CAPTURE_LAYER_STRIDE": str(CAPTURE_LAYER_STRIDE),
    "CAPTURE_MAX_BINS": str(CAPTURE_MAX_BINS),
    "HF_HOME": str(LOCAL_HF_HOME),
    "MUJOCO_GL": "egl",
    "PYOPENGL_PLATFORM": "egl",
    "MPLBACKEND": "Agg",
    "PYTHONUNBUFFERED": "1",
})

cmd = ["bash", str(GROOT_RUN / "scripts/run_groot_libero.sh"), SUITE, str(EPISODES)]
launcher_log = run_dir / "colab_launcher.log"
ANSI_RE = re.compile(r"\x1b\[[0-?]*[ -/]*[@-~]")


def eval_format_duration(seconds):
    if seconds is None or seconds != seconds or seconds < 0:
        return "unknown"
    seconds = int(seconds)
    hours, rem = divmod(seconds, 3600)
    minutes, secs = divmod(rem, 60)
    if hours:
        return f"{hours}h{minutes:02d}m{secs:02d}s"
    if minutes:
        return f"{minutes}m{secs:02d}s"
    return f"{secs}s"


def eval_path_size_bytes(path: Path) -> int:
    total = 0
    if not path.exists():
        return 0
    for root, dirs, files in os.walk(path):
        dirs[:] = [name for name in dirs if not Path(root, name).is_symlink()]
        for name in files:
            try:
                total += Path(root, name).lstat().st_size
            except OSError:
                pass
    return total


def tail_text(path: Path, max_bytes: int = 240_000) -> str:
    if not path.exists():
        return ""
    with path.open("rb") as handle:
        try:
            handle.seek(-max_bytes, os.SEEK_END)
        except OSError:
            handle.seek(0)
        return handle.read().decode("utf-8", errors="replace")


def task_id_count(raw: str):
    raw = raw.strip()
    if not raw:
        return None
    try:
        value = ast.literal_eval(raw)
    except Exception:
        return None
    if isinstance(value, int):
        return 1
    if isinstance(value, (list, tuple, set)):
        return len(value)
    return None


def expected_eval_count() -> int:
    suites = [item.strip() for item in SUITE.split(",") if item.strip()]
    per_suite = task_id_count(TASK_IDS)
    if per_suite is None:
        per_suite = 10
    return max(1, len(suites) * max(1, per_suite) * max(1, int(EPISODES)))


def print_eval_progress(start_time: float, final: bool = False, failed: bool = False):
    text = tail_text(launcher_log) + "\n" + tail_text(run_dir / "run.log") + "\n" + tail_text(run_dir / "server.log")
    videos = list((run_dir / "videos").glob("**/*.mp4")) if (run_dir / "videos").exists() else []
    expected = expected_eval_count()
    events = run_dir / "activation_capture" / "events.jsonl"
    chunks = 0
    if events.exists():
        chunks = tail_text(events, max_bytes=max(events.stat().st_size, 1)).count('"type":"chunk_start"')
    last = ""
    for line in reversed([line.strip() for line in ANSI_RE.sub("", text).splitlines() if line.strip()]):
        last = line[-220:]
        break
    label = "FAIL" if failed else ("DONE" if final else "HEARTBEAT")
    print(
        f"[{label}] elapsed={eval_format_duration(time.time() - start_time)} "
        f"videos={len(videos)}/{expected} size={eval_path_size_bytes(run_dir)} "
        f"chunks={chunks} last={last}",
        flush=True,
    )


print("$", " ".join(cmd), flush=True)
start = time.time()
with launcher_log.open("w", encoding="utf-8") as handle:
    proc = subprocess.Popen(
        cmd,
        cwd=str(LOCAL_REPO),
        env=run_env,
        stdout=handle,
        stderr=subprocess.STDOUT,
        text=True,
    )
    stop = {"done": False}

    def heartbeat():
        while not stop["done"]:
            time.sleep(max(5, int(EVAL_PROGRESS_SECONDS)))
            if stop["done"]:
                break
            print_eval_progress(start)

    thread = threading.Thread(target=heartbeat, daemon=True)
    thread.start()
    returncode = proc.wait()
    stop["done"] = True
    thread.join(timeout=1)

if returncode != 0:
    print_eval_progress(start, final=True, failed=True)
    print(tail_text(run_dir / "server.log", 8000))
    print(tail_text(run_dir / "run.log", 8000))
    raise RuntimeError(f"GR00T eval failed with code {returncode}. See {launcher_log}")

print_eval_progress(start, final=True)
print("LATEST_RUN", LATEST_RUN)
if (run_dir / "eval_info.json").exists():
    print((run_dir / "eval_info.json").read_text()[:2000])


In [ ]:
# @title Persist Outputs And Caches Back To Drive

import shutil
import subprocess
from pathlib import Path


def rsync_tree(src: Path, dest: Path, reset: bool = False):
    dest.mkdir(parents=True, exist_ok=True)
    cmd = ["rsync", "-a", "--info=progress2"]
    if reset:
        cmd.append("--delete")
    cmd.extend([f"{src}/", f"{dest}/"])
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)


rsync_tree(LOCAL_REPO / "outputs", DRIVE_OUTPUTS, reset=False)
if not HF_OFFLINE and LOCAL_HF_HOME.exists() and any(LOCAL_HF_HOME.iterdir()):
    archive = DRIVE_ARCHIVES / "groot_hf_home.tar"
    print("Updating", archive, flush=True)
    subprocess.run(f"tar -cf - -C {LOCAL_HF_HOME} . | pv > {archive}", shell=True, check=True)
if (ISAAC_ROOT / "checkpoints/GR00T-N1.7-LIBERO").exists() and (not HF_OFFLINE):
    archive = DRIVE_ARCHIVES / "groot_ckpt.tar"
    print("Updating", archive, flush=True)
    subprocess.run(
        f"tar -cf - -C {ISAAC_ROOT / 'checkpoints'} GR00T-N1.7-LIBERO | pv > {archive}",
        shell=True,
        check=True,
    )
print("Persisted outputs to", DRIVE_OUTPUTS / "eval" / "groot_libero")


In [ ]:
# @title Display Summary, Videos, And Activation Report Inline

import json
import subprocess
from pathlib import Path
from IPython.display import display, Video, Markdown, Image, HTML

run_dir = LATEST_RUN
info_path = run_dir / "eval_info.json"
info = json.loads(info_path.read_text()) if info_path.exists() else {}
overall = info.get("overall_success_rate", info.get("overall", {}))

display(Markdown(f"""
### Latest GR00T run

`{run_dir}`

- success: `{overall}`
- tasks: `{len(info.get('per_task', []))}`
- n_action_steps: `{info.get('n_action_steps', N_ACTION_STEPS)}`
"""))

videos = sorted((run_dir / "videos").glob("**/*.mp4"))
print("rollout videos:", len(videos))
if videos:
    display(Video(str(videos[0]), embed=True, width=720))

if CAPTURE_ACTIVATIONS:
    analysis_dir = run_dir / "analysis"
    analysis_dir.mkdir(parents=True, exist_ok=True)
    if GENERATE_DIAGNOSTIC_VIDEO:
        subprocess.run([
            str(GROOT_PYTHON), str(GROOT_RUN / "scripts/make_groot_analysis_video.py"),
            "--run", str(run_dir),
            "--task-id", str(ANALYSIS_TASK_ID),
            "--n-action-steps", str(N_ACTION_STEPS),
            "--preview-frame", "30",
        ], cwd=str(GROOT_RUN), check=True)
        previews = sorted(analysis_dir.glob("*_frame0030.png"))
        analysis_videos = sorted(analysis_dir.glob("*.mp4"))
        if previews:
            display(Markdown("### Four-panel diagnostic preview"))
            display(Image(filename=str(previews[-1]), width=1000))
        if analysis_videos:
            display(Video(str(analysis_videos[-1]), embed=True, width=900))

    subprocess.run([
        str(GROOT_PYTHON), str(GROOT_RUN / "scripts/make_groot_colab_report.py"),
        "--run", str(run_dir),
        "--task-id", str(ANALYSIS_TASK_ID),
        "--episode", "0",
        "--n-action-steps", str(N_ACTION_STEPS),
        "--max-rows", str(REPORT_MAX_ROWS),
    ], cwd=str(GROOT_RUN), check=True)
    report_dir = analysis_dir / f"task_{ANALYSIS_TASK_ID}_episode_0_colab_report"
    manifest_path = report_dir / f"task_{ANALYSIS_TASK_ID}_episode_0_manifest.json"
    manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
    display(Markdown("""
### Granular Investigation Report

Same interactive controls as the Pi0.5 notebook: family, metric, layer, optional action overlay.
Chunk rows are GR00T policy calls. The sim executes the first 8 actions, then replans.
Families: `vision` (backbone vision), `prefix` (backbone language), `expert` (DiT action head), `projection`.
"""))
    interactive_path = manifest.get("interactive_html")
    if interactive_path and Path(interactive_path).exists():
        display(HTML(Path(interactive_path).read_text()))
    else:
        for key, width in [("chunk_matrix", 1600), ("family_heatmaps", 1100), ("expert_layers_grid", 1100)]:
            path = manifest.get(key)
            if path and Path(path).exists():
                display(Markdown(f"#### {key.replace('_', ' ').title()}"))
                display(Image(filename=str(path), width=width))
    layer_graphs = [Path(p) for p in manifest.get("expert_layer_graphs", [])]
    if DISPLAY_INDIVIDUAL_LAYER_GRAPHS and layer_graphs:
        display(Markdown("#### Individual Expert Layer Graphs"))
        for path in layer_graphs[:max(0, int(LAYER_GRAPH_LIMIT))]:
            if path.exists():
                display(Image(filename=str(path), width=900))
    rsync_tree(LOCAL_REPO / "outputs", DRIVE_OUTPUTS, reset=False)
else:
    display(Markdown("Activation report skipped because `CAPTURE_ACTIVATIONS=False`."))


## Language intervention probe

This is not a second eval, and a single swapped-prompt inference is not a probe.

It resets one LIBERO task and, at every replan, asks GR00T for two 8-step chunks on the **same** cameras and state:

1. original task language
2. `PROBE_LANGUAGE`

The robot executes the original-language chunk so the scene follows the real task. The table is the difference those 8 numbers make when only the language changes. Set `PROBE_LANGUAGE` to something other than the official instruction; the default here targets the cookie box.

`PROBE_N_REPLANS` is how many planning windows to compare (4 windows = 32 executed steps). Low-level or out-of-distribution prompts can look like noise. This is not an official success metric.


In [ ]:
# @title Run Language Intervention Probe

import json
import os
import sys
import time
import subprocess
from pathlib import Path
from IPython.display import display, Markdown, Image

sys.path.insert(0, str(GROOT_RUN / "scripts"))
from resolve_tasks import resolve

task_map = json.loads((GROOT_RUN / "task_map.json").read_text())
selected = resolve(task_map, PROBE_SUITE, str(int(PROBE_TASK_ID)))

env_name = selected[0]["env_name"]
probe_out = LOCAL_REPO / "outputs/probes" / PROBE_SUITE / f"task_{PROBE_TASK_ID}"
probe_out.mkdir(parents=True, exist_ok=True)

# Reuse a short-lived server if the eval server is already gone.
probe_env = os.environ.copy()
probe_env.update({
    "MUJOCO_GL": "egl",
    "PYOPENGL_PLATFORM": "egl",
    "MPLBACKEND": "Agg",
    "HF_HOME": str(LOCAL_HF_HOME),
})

server_log = probe_out / "probe_server.log"
server = subprocess.Popen(
    [
        str(GROOT_PYTHON), "-u", str(ISAAC_ROOT / "gr00t/eval/run_gr00t_server.py"),
        "--model-path", str(model_dir),
        "--embodiment-tag", EMBODIMENT_TAG,
        "--use-sim-policy-wrapper",
        "--host", "127.0.0.1",
        "--port", "5556",
    ],
    cwd=str(ISAAC_ROOT),
    env=probe_env,
    stdout=server_log.open("w"),
    stderr=subprocess.STDOUT,
)
try:
    ready = False
    for _ in range(300):
        if server_log.exists() and "listening on" in server_log.read_text(errors="replace"):
            ready = True
            break
        if server.poll() is not None:
            break
        time.sleep(2)
    if not ready:
        raise RuntimeError(f"Probe server failed. See {server_log}")
    subprocess.run(
        [
            str(LIBERO_PYTHON), "-u", str(GROOT_RUN / "scripts/run_groot_prompt_probe.py"),
            "--env-name", env_name,
            "--language", PROBE_LANGUAGE,
            "--out-dir", str(probe_out),
            "--seed", str(PROBE_SEED),
            "--policy-client-host", "127.0.0.1",
            "--policy-client-port", "5556",
            "--n-action-steps", str(N_ACTION_STEPS),
            "--n-replans", str(int(globals().get("PROBE_N_REPLANS", 4))),
        ],
        cwd=str(ISAAC_ROOT),
        env=probe_env,
        check=True,
    )
finally:
    server.terminate()
    try:
        server.wait(timeout=20)
    except Exception:
        server.kill()

compare_path = probe_out / "compare.md"
if compare_path.exists():
    display(Markdown(compare_path.read_text(encoding="utf-8")))
else:
    display(Markdown(f"### Language probe `{PROBE_LANGUAGE}`"))
for name in ["render.png", "observation_images_image.png", "observation_images_image2.png"]:
    path = probe_out / name
    if path.exists():
        display(Image(filename=str(path), width=420))
rsync_tree(LOCAL_REPO / "outputs", DRIVE_OUTPUTS, reset=False)
print("Probe artifacts:", probe_out)
